In [1]:
# Import libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# Download the data files
!rm *.csv
!wget -q https://www.cs.utexas.edu/~kiat/datasets/Revised_Final_Data/Station1_Revised_Final_Data.csv
!wget -q https://www.cs.utexas.edu/~kiat/datasets/Revised_Final_Data/Station2_Revised_Final_Data.csv
!wget -q https://www.cs.utexas.edu/~kiat/datasets/Revised_Final_Data/Station3_Revised_Final_Data.csv
!wget -q https://www.cs.utexas.edu/~kiat/datasets/Revised_Final_Data/Station4_Revised_Final_Data.csv
!wget -q https://www.cs.utexas.edu/~kiat/datasets/Revised_Final_Data/Station5_Revised_Final_Data.csv
!wget -q https://www.cs.utexas.edu/~kiat/datasets/Revised_Final_Data/Station6_Revised_Final_Data.csv

In [3]:
# List of stations
stations = {
    'Station1_Revised_Final_Data.csv': 'Station 1',
    'Station2_Revised_Final_Data.csv': 'Station 2',
    'Station3_Revised_Final_Data.csv': 'Station 3',
    'Station4_Revised_Final_Data.csv': 'Station 4',
    'Station5_Revised_Final_Data.csv': 'Station 5',
    'Station6_Revised_Final_Data.csv': 'Station 6'
}

In [4]:
# Load and process data for each station
def load_rainfall_data(file, station_name, time = 'year', pivot = False):
    df = pd.read_csv(file, index_col=0, parse_dates=True)
    df.reset_index(inplace=True)
    df.rename(columns={'index': 'Date'}, inplace=True)
    
    if time == 'year':
        df['Group'] = df['Date'].dt.year
    elif time == 'month':
        df['Group'] = df['Date'].dt.to_period('M').astype(str)
    elif time == 'month_matrix':
        df['Year'] = df['Date'].dt.year
        df['Month'] = df['Date'].dt.month_name()
        df['Month_num'] = df['Date'].dt.month
        monthly_totals = df.groupby(['Year', 'Month', 'Month_num'])['Ppt'].sum().reset_index()
        matrix = monthly_totals.pivot_table(index='Month', columns='Year', values='Ppt')
        ordered_months = ['January', 'February', 'March', 'April', 'May', 'June',
                          'July', 'August', 'September', 'October', 'November', 'December']
        return matrix.reindex(ordered_months)
    else:
        raise ValueError("`by` must be 'year', 'month', or 'month_matrix'")

    totals = df.groupby('Group')['Ppt'].sum().reset_index()
    totals.columns = [time.title(), station_name]

    if pivot:
        return totals.set_index(time.title())
    return totals

In [5]:
# Merge all data into rainfall per year
rain_year_df = None
for file, name in stations.items():
    df = load_rainfall_data(file, name, time = 'year')
    rain_year_df = df if rain_year_df is None else pd.merge(rain_year_df, df, on = 'Year', how = 'outer')
rain_year_df.set_index('Year', inplace=True)

In [6]:
rain_year_df.head()

,Station 1,Station 2,Station 3,Station 4,Station 5,Station 6
Year,,,,,,
2015,488.400000,800.38,942.71,755.58,810.71,731.69
2016,476.870000,864.62,890.41,926.76,814.81,638.30
2017,604.050000,528.76,649.38,629.81,620.43,441.23
2018,891.553044,692.70,740.80,659.74,654.00,798.36
2019,480.710000,501.91,558.57,485.00,441.59,458.35


In [7]:
# Merge all data into rainfall per month
rain_month_df = None
for file, name in stations.items():
    df = load_rainfall_data(file, name, time = 'month')
    rain_month_df = df if rain_month_df is None else pd.merge(rain_month_df, df, on = 'Month', how = 'outer')
rain_month_df.set_index('Month', inplace=True)

In [8]:
rain_month_df.head(10)

,Station 1,Station 2,Station 3,Station 4,Station 5,Station 6
Month,,,,,,
2015-01,53.48,76.89,75.15,56.00,71.56,42.87
2015-02,4.29,5.05,5.33,4.79,3.30,3.04
2015-03,57.82,48.13,53.26,60.07,61.65,49.46
2015-04,58.92,64.74,74.66,65.52,99.30,55.11
2015-05,219.44,245.06,264.86,208.24,230.35,243.82
2015-06,56.63,50.79,89.64,73.63,26.89,41.14
2015-07,28.19,2.03,17.27,0.00,0.00,5.58
2015-08,1.01,4.06,15.74,20.32,10.16,18.79
2015-09,8.62,16.00,14.48,5.82,10.41,6.60


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=8e560f6b-1726-4372-a865-7294f65e8413' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>